# Task 3 — Model 1: ARIMA (statistical baseline)

One-step-ahead forecasts of internet traffic for three Milan grid cells during the **test week 16–22 Dec 2013**. Training uses Nov 1 – Dec 15 only (test week is never seen during fit).

Neural models will be added in separate notebooks later.

In [ ]:
# Libraries
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA

In [ ]:
# Paths and date boundaries (assignment test week)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "milan_internet_traffic.parquet"
FOCUS_PATH = PROJECT_ROOT / "data" / "processed" / "task2_focus_squares.csv"
FIG_DIR = PROJECT_ROOT / "figures" / "task3_arima"
OUT_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_START = "2013-11-01"
TRAIN_END = "2013-12-16"      # train: [start, end)
TEST_START = "2013-12-16"
TEST_END = "2013-12-23"      # test week: [start, end)

# ARIMA order chosen after Task 2 ACF/PACF (short AR, differencing, MA)
ARIMA_ORDER = (2, 1, 2)

In [ ]:
# Three geographical areas from Task 2
focus = pd.read_csv(FOCUS_PATH)
SQUARE_IDS = focus["square_id"].tolist()
LABELS = dict(zip(focus["square_id"], focus["label"]))
print(focus)

## Model description (for report)

**ARIMA(p,d,q)** models the (differenced) series as autoregressive + moving-average noise. We use **ARIMA(2,1,2)**:
- **Input history:** all observed 10-minute traffic values from Nov 1 up to and including time \(t\) (regular grid, missing slots = 0).
- **Preprocessing:** filter Italy/internet field (done in Task 1); reindex to 10-minute frequency; no extra normalization (traffic is already positive CDR counts).
- **Training:** fit once on the training window; during the test week, **one-step-ahead** forecasts use the fitted state updated with each **actual** \(y_t\) (`append`, no refit) — fair evaluation with true history.
- **Seasonality note:** Task 2 showed daily cycles (lag 144). Full SARIMA(·,·,·)(·,·,·,144) is very slow; ARIMA(2,1,2) is the statistical baseline; seasonal structure is partly absorbed by differencing and MA terms.

In [ ]:
# Load only the three focus squares once (avoids reloading ~89M rows per fit)
traffic = pd.read_parquet(PARQUET_PATH)
traffic = traffic[traffic["square_id"].isin(SQUARE_IDS)]
print(f"Subset rows: {len(traffic):,}")

In [ ]:
def series_for_square(square_id: int, t0: str, t1: str) -> pd.Series:
    """10-minute traffic for one square in [t0, t1)."""
    chunk = traffic[(traffic["square_id"] == square_id) & (traffic["time"] >= t0) & (traffic["time"] < t1)]
    s = chunk.set_index("time")["internet"].sort_index()
    full_idx = pd.date_range(t0, t1, freq="10min", inclusive="left")
    return s.reindex(full_idx, fill_value=0).astype(float)

In [ ]:
def fit_arima(train: pd.Series):
    """Fit ARIMA on training window; return fitted results object."""
    model = ARIMA(train, order=ARIMA_ORDER)
    return model.fit()

In [ ]:
def forecast_test_week(fitted, test: pd.Series) -> pd.Series:
    """One-step-ahead predictions for each test interval (history updated with actuals)."""
    preds = []
    res = fitted
    for y in test.values:
        step = res.forecast(1)
        preds.append(float(step.iloc[0]))
        res = res.append([y], refit=False)
    return pd.Series(preds, index=test.index, name="predicted")

In [ ]:
def error_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """MAE, MAPE (%), RMSE for report tables."""
    err = y_true - y_pred
    mae = np.mean(np.abs(err))
    # avoid division by zero in MAPE
    denom = np.where(y_true == 0, np.nan, y_true)
    mape = np.nanmean(np.abs(err / denom)) * 100
    rmse = np.sqrt(np.mean(err ** 2))
    return {"MAE": mae, "MAPE": mape, "RMSE": rmse}

## Optional: quick order check on highest-traffic square (experimentation)

Compare small candidate orders via AIC on the training set only.

In [ ]:
# Run only on busiest cell; keep (2,1,2) unless AIC clearly prefers another
sq0 = SQUARE_IDS[0]
train_probe = series_for_square(sq0, TRAIN_START, TRAIN_END)
candidates = [(1, 1, 1), (2, 1, 2), (3, 1, 2)]
aic_rows = []
for order in candidates:
    fit = ARIMA(train_probe, order=order).fit()
    aic_rows.append({"order": order, "AIC": fit.aic})
display(pd.DataFrame(aic_rows).sort_values("AIC"))

## Train, forecast, and evaluate — all three areas

In [ ]:
results = {}      # per-square predictions
metrics = []      # per-square errors
timings = []      # train / forecast seconds

for sq in SQUARE_IDS:
    train = series_for_square(sq, TRAIN_START, TRAIN_END)
    test = series_for_square(sq, TEST_START, TEST_END)

    t0 = time.perf_counter()
    fitted = fit_arima(train)
    train_sec = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred = forecast_test_week(fitted, test)
    forecast_sec = time.perf_counter() - t1

    m = error_metrics(test.values, pred.values)
    m.update({"square_id": sq, "label": LABELS[sq], "model": "ARIMA(2,1,2)"})
    metrics.append(m)
    timings.append({"square_id": sq, "train_sec": train_sec, "forecast_sec": forecast_sec})
    results[sq] = pd.DataFrame({"actual": test, "predicted": pred})
    print(f"Square {sq} ({LABELS[sq]}): train {train_sec:.1f}s, forecast {forecast_sec:.1f}s, MAE={m['MAE']:.2f}")

In [ ]:
# Metrics table (one row per area for ARIMA)
metrics_df = pd.DataFrame(metrics)[["label", "square_id", "model", "MAE", "MAPE", "RMSE"]]
display(metrics_df)
metrics_df.to_csv(OUT_DIR / "arima_metrics_by_area.csv", index=False)

In [ ]:
# Timing table + hardware note for the report
timing_df = pd.DataFrame(timings)
timing_df["hardware"] = platform.platform()
timing_df["python"] = platform.python_version()
display(timing_df)
timing_df.to_csv(OUT_DIR / "arima_timings.csv", index=False)

## Plots — actual vs predicted (test week)

Three figures for ARIMA (nine total after all models are added).

In [ ]:
for sq in SQUARE_IDS:
    df = results[sq]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df.index, df["actual"], label="Actual", linewidth=0.9)
    ax.plot(df.index, df["predicted"], label="ARIMA pred.", linewidth=0.9, alpha=0.85)
    ax.set_title(f"ARIMA(2,1,2) — {LABELS[sq]} (square {sq}) — Dec 16–22, 2013")
    ax.set_xlabel("Time")
    ax.set_ylabel("Internet traffic")
    ax.legend()
    plt.tight_layout()
    fname = FIG_DIR / f"arima_square_{sq}.png"
    plt.savefig(fname, dpi=150)
    plt.show()

In [ ]:
# Save predictions for later comparison with neural models
for sq, df in results.items():
    path = OUT_DIR / f"arima_predictions_square_{sq}.csv"
    df.to_csv(path)
    print(f"Saved {path.name}")

## Notes for report (ARIMA section)

**Evaluation metrics**
- MAE: \(\frac{1}{n}\sum |y_i - \hat{y}_i|\)
- MAPE: \(\frac{100}{n}\sum |y_i - \hat{y}_i|/|y_i|\) (zeros ignored)
- RMSE: \(\sqrt{\frac{1}{n}\sum (y_i - \hat{y}_i)^2}\)

**Timing:** `train_sec` = single `fit()` on Nov–Dec 15; `forecast_sec` = rolling one-step loop over the test week.

**Limitations / failure hints:** Large nightly/morning swings and holiday effects (Dec) are hard for non-seasonal ARIMA; errors often spike at **daily regime changes** (compare overlay plots). Full SARIMA or sequence models may improve daily seasonality.